In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
import statsmodels.stats.multicomp as mc
from IPython.display import display

## 1. Leitura dos dados

In [ ]:
PASTA_PRECIP = os.path.join("..", "DATA", "CSV_regioes")   # precipitação
PASTA_ACUDES = os.path.join("..", "DATA", "CSV_acudes")    # volume dos açudes

def ler_pasta(pasta, sep=";", **kwargs):
    lista = []
    for nome in sorted(os.listdir(pasta)):
        if nome.lower().endswith(".csv"):
            df = pd.read_csv(os.path.join(pasta, nome), sep=sep, **kwargs)
            df["arquivo"] = os.path.splitext(nome)[0]
            lista.append(df)
    if not lista:
        raise FileNotFoundError(f"Nenhum CSV em: {pasta}")
    return lista

df_1 = pd.concat(ler_pasta(PASTA_PRECIP, decimal="."), ignore_index=True)
df_2 = pd.concat(ler_pasta(PASTA_ACUDES, decimal=","), ignore_index=True).rename(columns={"arquivo": "acude"})

display(df_1.head())
display(df_2.head())
print(df_1.dtypes, "\n")
print(df_2.dtypes)

## 2. Pré-processamento dos açudes

- `pct`: volume em % da capacidade máxima (limitado a 100%).
- `classe_acude`: classe ordinal do volume (≤10% muito crítico, ≤30% crítico, ≤50% alerta, ≤70% confortável, >70% muito confortável).
- `status_alerta`: **0** = muito crítico ou crítico (≤30%), **1** = acima de 30%.

In [ ]:
# Padroniza o nome do açude (o arquivo pode vir como 'olho_dagua' ou 'olhodgua')
df_2["acude"] = df_2["acude"].replace({"reserv_olho_dagua": "reserv_olhodgua"})

capacidade = {            # hm³
    "reserv_olhodgua":   19.210,
    "reserv_quixabinha": 31.780,
    "reserv_atalho":     72.550,
    "reserv_rosario":    47.220,
}

df_2["capacidade_maxima"] = df_2["acude"].map(capacidade).astype(float)
sem_capacidade = df_2.loc[df_2["capacidade_maxima"].isna(), "acude"].unique()
if len(sem_capacidade):
    print("ATENÇÃO: açudes sem capacidade cadastrada:", sem_capacidade)

df_2["pct"] = (df_2["Volume (hm³)"] / df_2["capacidade_maxima"] * 100).clip(upper=100)
pct = df_2["pct"]

condicoes = [
    (pct >= 0)  & (pct <= 10),
    (pct > 10)  & (pct <= 30),
    (pct > 30)  & (pct <= 50),
    (pct > 50)  & (pct <= 70),
    (pct > 70)  & (pct <= 100),
]
rotulos = ["muito_critico", "critico", "alerta", "confortavel", "muito_confortavel"]
df_2["classe_acude"] = np.select(condicoes, rotulos, default="sem_classe")
df_2.loc[pct.isna() | (df_2["classe_acude"] == "sem_classe"), "classe_acude"] = np.nan

# 0 = muito crítico/crítico (<=30%) | 1 = acima de 30%; NA onde não há volume
df_2["status_alerta"] = (pct > 30).astype("Int64").where(pct.notna())

print(df_2["classe_acude"].value_counts(dropna=False))
print(df_2["status_alerta"].value_counts(dropna=False))

## 3. Integração: açude × ponto de precipitação mais próximo

Cada açude é ligado ao ponto de chuva mais próximo (distância de Haversine, raio máximo de 50 km) e os dados são unidos pela data.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Distância em km entre pontos (fórmula de Haversine)."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

coordenadas_acudes = pd.DataFrame({
    "acude":     ["reserv_atalho", "reserv_rosario", "reserv_olhodgua", "reserv_quixabinha"],
    "lat_acude": [-8.75, -6.75, -6.25, -7.25],
    "lon_acude": [-38.25, -39.25, -36.25, -38.75],
})
RAIO_MAXIMO_KM = 50.0

# chave de data em texto, igual nas duas tabelas
df_1["data_str"] = pd.to_datetime(df_1["data"]).dt.strftime("%Y-%m-%d")
df_2["data_str"] = pd.to_datetime(df_2["DateTime"]).dt.strftime("%Y-%m-%d")

# remove linhas repetidas de precipitação (mesma data e mesmo ponto)
antes = len(df_1)
df_1 = df_1.drop_duplicates(subset=["data_str", "lat", "lon"]).reset_index(drop=True)
print(f"Linhas repetidas removidas em df_1: {antes - len(df_1)}")

pontos = df_1[["lat", "lon"]].drop_duplicates().reset_index(drop=True)
mapeamento = []
for _, ac in coordenadas_acudes.iterrows():
    dist = haversine_km(ac["lat_acude"], ac["lon_acude"], pontos["lat"], pontos["lon"])
    i = dist.idxmin()
    if dist.loc[i] <= RAIO_MAXIMO_KM:
        mapeamento.append({
            "acude": ac["acude"],
            "lat_precip_mais_proxima": pontos.loc[i, "lat"],
            "lon_precip_mais_proxima": pontos.loc[i, "lon"],
            "distancia_km": dist.loc[i],
        })
    else:
        print(f"Açude '{ac['acude']}' ignorado: ponto de chuva mais próximo a {dist.loc[i]:.1f} km (limite {RAIO_MAXIMO_KM} km).")

df_map = pd.DataFrame(mapeamento)
if df_map.empty:
    raise ValueError("Nenhum açude atendeu ao critério de distância.")

df_2_geo = df_2.merge(df_map, on="acude", how="inner")
df_unificado = df_2_geo.merge(
    df_1,
    left_on=["data_str", "lat_precip_mais_proxima", "lon_precip_mais_proxima"],
    right_on=["data_str", "lat", "lon"],
    how="inner",
)
print(f"Registros válidos (raio de {RAIO_MAXIMO_KM} km): {len(df_unificado)}")
print(df_unificado["acude"].value_counts())

df_final = df_unificado.rename(columns={
    "DateTime": "Data",
    "Volume (hm³)": "volume",
    "precip_mm": "precipitação",
    "classe_acude": "classe_volume_acude",
    "status_alerta": "bin_status_alerta",
})
cols = ["Data", "acude", "municipio", "estado", "volume", "pct",
        "precipitação", "classe_volume_acude", "bin_status_alerta"]
df_final = df_final[[c for c in cols if c in df_final.columns]].copy()

print(df_final.dtypes)
print(df_final.isna().sum())
display(df_final.head())

---
## QUESTÃO 2: teste de associação (nos dados originais)

**Natureza das variáveis**
- `precipitação` (mm): **quantitativa contínua** (preditora).
- `classe_volume_acude`: **qualitativa ordinal** (dependente).

Como a preditora é quantitativa e a dependente é qualitativa, compara-se a precipitação entre os grupos da qualitativa:

1. Descritiva e boxplot por grupo.
2. **ANOVA** (H0: a média da precipitação é igual em todas as classes de volume).
3. Pressupostos: **normalidade dos resíduos** (KS; H0: normais) e **homogeneidade de variâncias** (Levene; H0: variâncias iguais).
4. Se algum pressuposto for rejeitado, usa-se o **Kruskal-Wallis** (H0: a localização da precipitação é a mesma em todos os grupos).

Critério de decisão: α = 5%.

In [ ]:
quanti, quali = "precipitação", "classe_volume_acude"
ALPHA = 0.05

ordem = ["muito_critico", "critico", "alerta", "confortavel", "muito_confortavel"]
ordem = [o for o in ordem if o in df_final[quali].dropna().unique()]

dq = df_final[[quanti, quali]].dropna().copy()
dq[quali] = pd.Categorical(dq[quali], categories=ordem, ordered=True)
print("n =", len(dq))

# 1) Descritiva por grupo
print(dq.groupby(quali, observed=True)[quanti].describe().round(2))

ax = dq.boxplot(column=quanti, by=quali, figsize=(7, 4), showmeans=True)
ax.set_ylabel("Precipitação (mm)")
ax.set_xlabel("Classe do volume do açude")
plt.suptitle("")
plt.title("Precipitação por classe de volume")
plt.show()

In [ ]:
# 2) ANOVA
d_aov = dq.rename(columns={quanti: "precip", quali: "classe"})
model = ols("precip ~ C(classe)", data=d_aov).fit()
aov_table = sm.stats.anova_lm(model, typ=2)
print(aov_table, "\n")

# 3a) Normalidade dos resíduos (H0: normais)
ks = stats.kstest(model.resid, stats.norm.cdf, args=(model.resid.mean(), model.resid.std()))
print("KS resíduos:", ks)

# 3b) Homogeneidade de variâncias (H0: variâncias iguais)
grupos = [g[quanti].dropna() for _, g in dq.groupby(quali, observed=True)]
lev = stats.levene(*grupos)
print("Levene:", lev)

normal = ks.pvalue >= ALPHA
homogeneo = lev.pvalue >= ALPHA
print("\nNormalidade dos resíduos:", "não rejeitada" if normal else "REJEITADA")
print("Homogeneidade de variâncias:", "não rejeitada" if homogeneo else "REJEITADA")
usar_anova = normal and homogeneo
print("Teste adequado:", "ANOVA" if usar_anova else "Kruskal-Wallis")

In [ ]:
# 4) Kruskal-Wallis
H, p_kw = stats.kruskal(*grupos)
n = sum(len(g) for g in grupos)
eps2 = H / (n - 1)   # tamanho de efeito: ~0,01 pequeno | ~0,06 médio | ~0,14 grande
print(f"Kruskal-Wallis: H = {H:.2f}, p = {p_kw:.4g}, epsilon² = {eps2:.4f}")
print("Decisão:", "rejeita H0 (há associação)" if p_kw < ALPHA else "não rejeita H0 (sem evidência de associação)")

print()
print(dq.groupby(quali, observed=True)[quanti].agg(["count", "mean", "median"]).round(2))

In [ ]:
# Comparações múltiplas (post hoc): Tukey após ANOVA | Dunn após Kruskal-Wallis
if usar_anova:
    comp = mc.MultiComparison(dq[quanti], dq[quali].astype(str))
    print(comp.tukeyhsd().summary())
else:
    try:
        import scikit_posthocs as sp          # pip install scikit-posthocs
        print(sp.posthoc_dunn(dq, val_col=quanti, group_col=quali, p_adjust="bonferroni").round(4))
    except ImportError:
        print("Para o post hoc de Dunn: pip install scikit-posthocs")

### Variante: dependente binária (`bin_status_alerta`)

Com dois grupos, o fluxo é o mesmo (o script05 usa `kruskal` com dois grupos); o Mann-Whitney é o equivalente.

In [ ]:
d2 = df_final[[quanti, "bin_status_alerta"]].dropna()
g0 = d2.loc[d2["bin_status_alerta"] == 0, quanti]
g1 = d2.loc[d2["bin_status_alerta"] == 1, quanti]
print(f"n (alerta=0): {len(g0)} | n (alerta=1): {len(g1)}")
print(f"Mediana (alerta=0): {g0.median():.2f} | Mediana (alerta=1): {g1.median():.2f}")
print("Levene:", stats.levene(g0, g1))
print("Kruskal-Wallis:", stats.kruskal(g0, g1))
print("Mann-Whitney:", stats.mannwhitneyu(g0, g1))

---
## QUESTÃO 3: transformação da quantitativa em qualitativa (intervalos)

Só entram as quantitativas **associadas estatisticamente à dependente** na Questão 2. A precipitação foi significativa, então é discretizada.

**Critérios dos intervalos**
- Cortes por **intensidade de chuva** (contexto meteorológico), em vez de quantis: a maioria dos dias tem 0 mm, e os quantis repetiriam limites.
- **Faixa própria para 0 mm** (dia seco).
- Faixas **mutuamente exclusivas** e que **cobrem todo o domínio** (limites com `>` e `<=`; última faixa aberta, > 50 mm).

**Por que transformar:** facilita a interpretação ("chuva forte" em vez de mm exatos), reduz o peso de valores extremos e permite usar a variável em modelos para dados categóricos (como a rede bayesiana).

In [ ]:
p_mm = df_final["precipitação"]

condicoes = [
    (p_mm == 0.0),
    (p_mm > 0.0)  & (p_mm <= 1.0),
    (p_mm > 1.0)  & (p_mm <= 10.0),
    (p_mm > 10.0) & (p_mm <= 20.0),
    (p_mm > 20.0) & (p_mm <= 30.0),
    (p_mm > 30.0) & (p_mm <= 40.0),
    (p_mm > 40.0) & (p_mm <= 50.0),
    (p_mm > 50.0),
]
rotulos = ["Sem_Precipitação", "Chuvisco", "Fraca", "Moderada",
           "Moderada_a_Forte", "Forte", "Muito_Forte", "Violenta"]

classe = np.select(condicoes, rotulos, default="sem_classe")
df_final["classe_precipitacao"] = pd.Categorical(classe, categories=rotulos, ordered=True)
df_final.loc[p_mm.isna(), "classe_precipitacao"] = np.nan

# versão binária: choveu (1) ou não (0)
df_final["bin_choveu"] = (p_mm > 0.0).astype("Int64").where(p_mm.notna())

print("Valores sem classe:", df_final["classe_precipitacao"].isna().sum())
freq = df_final["classe_precipitacao"].value_counts().sort_index()
display(pd.DataFrame({"n": freq, "%": (freq / freq.sum() * 100).round(2)}))

In [ ]:
tab = pd.crosstab(df_final["classe_precipitacao"], df_final["classe_volume_acude"])
tab = tab.reindex(columns=[o for o in ordem if o in tab.columns])
display(tab)

chi2, p_chi, gl, esp = stats.chi2_contingency(tab)
n = tab.values.sum()
v = (chi2 / (n * (min(tab.shape) - 1))) ** 0.5

print(f"chi2 = {chi2:.2f}, gl = {gl}, p = {p_chi:.4g}, V de Cramér = {v:.3f}")
print(f"Células com frequência esperada <= 5: {(esp <= 5).sum()} de {esp.size}")
print("Decisão:", "rejeita H0 (há associação)" if p_chi < ALPHA else "não rejeita H0")

In [ ]:
# Caso 2x2: bin_choveu x bin_status_alerta (Fisher se houver frequência esperada baixa)
tab2 = pd.crosstab(df_final["bin_choveu"], df_final["bin_status_alerta"])
display(tab2)
chi2b, pb, glb, espb = stats.chi2_contingency(tab2)
print(f"Qui-quadrado: chi2 = {chi2b:.2f}, p = {pb:.4g}")
if (espb <= 5).any():
    print("Fisher exato:", stats.fisher_exact(tab2))